In [35]:
#r "nuget: ScottPlot, 5.0.19"

Installed Packages ScottPlot, 5.0.19

In [36]:
using System;
using System.Diagnostics;
using System.Linq;
using System.Threading;
using System.IO;
using ScottPlot;

In [37]:
public class DefiniteIntegralOptimized
{
    public static double Solve(double a, double b, Func<double, double> function, double step, int threadsnumber)
    {
        double result = 0.0;
        object lockObj = new object();
        double range = b - a;
        double stepSize = range / threadsnumber;

        Parallel.For(0, threadsnumber, new ParallelOptions { MaxDegreeOfParallelism = threadsnumber }, i =>
        {
            double threadStart = a + i * stepSize;
            double threadEnd = (i == threadsnumber - 1) ? b : threadStart + stepSize;
            double localResult = 0.0;

            for (double x = threadStart; x < threadEnd; x += step)
            {
                double next = Math.Min(x + step, threadEnd);
                double nextVal = function(next);
                localResult += (function(x) + nextVal) * (next - x) / 2.0;
            }

            lock (lockObj)
            {
                result += localResult;
            }
        });

        return result;
    }
}

 display("Метод вычисления определенного класса был оптимизирован. Вместо ручного управления потоков теперь используется пул-потоков (Parallel), что позволило увеличить производительность до 80%")

Метод вычисления определенного класса был оптимизирован. Вместо ручного управления потоков теперь используется пул-потоков (Parallel), что позволило увеличить производительность до 80%

In [38]:
double SingleThreadIntegral(double a, double b, Func<double, double> function, double step)
{
    double result = 0.0;
    for (double x = a; x < b; x += step)
    {
        double next = Math.Min(x + step, b);
        result += (function(x) + function(next)) * (next - x) / 2.0;
    }
    return result;
}

In [39]:
int a = -100;
int b = 100;
double[] steps = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
double accuracy = 1e-4;
double bestStep = steps[0];
double bestTime = double.MaxValue;
int iterations = 3;

foreach (var step in steps)
{
    double result = 0;
    Stopwatch sw = Stopwatch.StartNew();
    for (int i = 0; i < iterations; i++)
    {
        result = DefiniteIntegralOptimized.Solve(a, b, Math.Sin, step, 4);
    }
    sw.Stop();

    double avgTime = sw.Elapsed.TotalMilliseconds / iterations;

    if (Math.Abs(result) <= accuracy && avgTime < bestTime)
    {
        bestStep = step;
        bestTime = avgTime;
    }
}
display($"Оптимальный шаг: {bestStep}, среднее время: {bestTime:F2} мс");

Оптимальный шаг: 0.1, среднее время: 0.29 мс

In [40]:
int[] threadCounts = { 2, 4, 6, 8, 10, 12, 14, 16 };
double[] avgTimes = new double[threadCounts.Length];

for (int i = 0; i < threadCounts.Length; i++)
{
    double sumTime = 0;
    for (int j = 0; j < iterations; j++)
    {
        var sw = Stopwatch.StartNew();
        DefiniteIntegralOptimized.Solve(a, b, Math.Sin, bestStep, threadCounts[i]);
        sw.Stop();
        sumTime += sw.Elapsed.TotalMilliseconds;
    }
    avgTimes[i] = sumTime / (double)iterations;
}

int bestThreads = threadCounts[Array.IndexOf(avgTimes, avgTimes.Min())];
display($"Оптимальное число потоков: {bestThreads}");

Оптимальное число потоков: 12

In [41]:
double singleThreadTime = 0;
for (int j = 0; j < iterations; j++)
{
    var sw = Stopwatch.StartNew();
    SingleThreadIntegral(a, b, Math.Sin, bestStep);
    sw.Stop();
    singleThreadTime += sw.Elapsed.TotalMilliseconds;
}
singleThreadTime /= iterations;

double multiThreadTime = avgTimes[Array.IndexOf(threadCounts, bestThreads)];
double percentDiff = 100.0 * (singleThreadTime - multiThreadTime) / singleThreadTime;

string result = $@"Время однопоточной версии: {singleThreadTime:F2} мс
Время многопоточной версии: {multiThreadTime:F2} мс
Разница: {percentDiff:F2}%";

display(result);
File.WriteAllText("benchmark.txt", result);

Время однопоточной версии: 0.31 мс
Время многопоточной версии: 0.06 мс
Разница: 79.24%

In [42]:
var scottPlot = new ScottPlot.Plot();

scottPlot.Add.Scatter(avgTimes, threadCounts.Select(x => (double)x).ToArray());
scottPlot.XLabel("Время выполнения (мс)");
scottPlot.YLabel("Количество потоков");
scottPlot.Title("Производительность");
scottPlot.SavePng("benchmark.png", 800, 600);

scottPlot

ScottPlot.Plot PlottableList index value 0 ScottPlot.Plottables.Scatter Label <null> IsVisible True Axes ScottPlot.Axes XAxis ScottPlot.AxisPanels.BottomAxis Min=0.05147833333333332, Max=0.3210883333333333, Span=0.26961 Edge Bottom Width 0.26961 IsVisible True Range Min=0.05147833333333332, Max=0.3210883333333333, Span=0.26961 Min 0.05147833333333332 Max 0.3210883333333333 Center 0.1862833333333333 Span 0.26961 HasBeenSet True ToCoordinateRange CoordinateRange { Min = 0.05147833333333332, Max = 0.3210883333333333, Span = 0.26961, Center = 0.1862833333333333, IsReal = True } MinimumSize 0 MaximumSize 3.4028235E+38 Min 0.05147833333333332 Max 0.3210883333333333 TickGenerator ScottPlot.TickGenerators.NumericAutomatic Ticks [ ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick ... (34 more) ] MaxTickCount 10000 LabelFormatter System.Func`2[System.Double,System.String] Label ScottPlot.Label IsVisible True Text Время выполнения (мс) Alignment LowerCenter Rotation 0 ForeColor ScottPlot.Color BackColor ScottPlot.Color BorderColor ScottPlot.Color BorderWidth 1 FontName Segoe UI FontSize 16 Bold True UseCachedTypefaces True Italic False AntiAlias True Padding 0 PointSize 0 PointFilled False PointColor ScottPlot.Color OffsetX 0 OffsetY 0 ShowDebugInformation False FrameLineStyle ScottPlot.LineStyle Width 1 Color ScottPlot.Color Pattern Solid IsVisible True AntiAlias True MajorTickStyle ScottPlot.TickMarkStyle Length 4 Width 1 Color ScottPlot.Color MinorTickStyle ScottPlot.TickMarkStyle Length 2 Width 1 Color ScottPlot.Color TickLabelStyle ScottPlot.Label IsVisible True Text 0.3 Alignment MiddleCenter Rotation 0 ForeColor ScottPlot.Color BackColor ScottPlot.Color BorderColor ScottPlot.Color BorderWidth 1 FontName Segoe UI FontSize 12 Bold False UseCachedTypefaces True Italic False AntiAlias True Padding 0 PointSize 0 PointFilled False PointColor ScottPlot.Color OffsetX 0 OffsetY 0 YAxis ScottPlot.AxisPanels.LeftAxis Min=0.95, Max=17.05, Span=16.1 Edge Left Height 16.1 IsVisible True Range Min=0.95, Max=17.05, Span=16.1 Min 0.95 Max 17.05 Center 9 Span 16.1 HasBeenSet True ToCoordinateRange CoordinateRange { Min = 0.95, Max = 17.05, Span = 16.1, Center = 9, IsReal = True } MinimumSize 0 MaximumSize 3.4028235E+38 Min 0.95 Max 17.05 TickGenerator ScottPlot.TickGenerators.NumericAutomatic Ticks [ ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick, ScottPlot.Tick ... (61 more) ] MaxTickCount 10000 LabelFormatter System.Func`2[System.Double,System.String] Label ScottPlot.Label IsVisible True Text Количество потоков Alignment UpperCenter Rotation -90 ForeColor ScottPlot.Color BackColor ScottPlot.Color BorderColor ScottPlot.Color BorderWidth 1 FontName Segoe UI FontSize 16 Bold True UseCachedTypefaces True Italic False AntiAlias True Padding 0 PointSize 0 PointFilled False PointColor ScottPlot.Color OffsetX 0 OffsetY 0 ShowDebugInformation False FrameLineStyle ScottPlot.LineStyle Width 1 Color ScottPlot.Color Pattern Solid IsVisible True AntiAlias True MajorTickStyle ScottPlot.TickMarkStyle Length 4 Width 1 Color ScottPlot.Color MinorTickStyle ScottPlot.TickMarkStyle Length 2 Width 1 Color ScottPlot.Color TickLabelStyle ScottPlot.Label IsVisible True Text 17 Alignment MiddleRight Rotation 0 ForeColor ScottPlot.Color BackColor ScottPlot.Color BorderColor ScottPlot.Color BorderWidth 1 FontName Segoe UI FontSize 12 Bold False UseCachedTypefaces True Italic False AntiAlias True Padding 0 PointSize 0 PointFilled False PointColor ScottPlot.Col